### Dimensionality and other prototype requirements

Beyond the target composition, the prototype search can be constrained by other fields stored in MPDD via the `other_targets` argument of `generate_MPDD`. The dictionary is merged into the MongoDB `$match` stage of the aggregation pipeline, so multiple keys are AND-ed together and any MongoDB query operator (e.g., `$gte`, `$in`, `$exists`) is supported.

Each `other_targets` value also produces a distinct output directory (a short hash of the filter is folded into the folder name), so reruns with the same filter update in place while different filters never overwrite each other. A `_query.json` sidecar is written next to each directory recording the exact inputs.

A few examples follow. Note that the exact field names depend on the MPDD schema — the keys below match common conventions, but verify against your collection if a query returns no results.

First, start by importing CrystALLSearch, initialize, and connect with MPDD.

In [ ]:
%%capture
from crystall import CrystALLSearch

In [ ]:
crystallSearch = CrystALLSearch("BaSO4")

In [ ]:
from pymongo import MongoClient
import os; client_string = os.getenv('MONGODB_URI')

MPDD_database = MongoClient(client_string)['MPDD']
crystallSearch.connect_MPDD(MPDD_database)

**Example 1 — restrict to confidently-2D structures.** Require `material.dimensionality == 2` *and* a dimensionality-agreement score of at least `0.75` (high confidence in the 2D label). Operators like `$gte` are passed through directly to MongoDB; the two keys are AND-ed together.

In [ ]:
other_targets = {
    "material.dimensionality": 2,
    "material.dimensionalityAgreement": {"$gte": 0.75},
}
crystallSearch.generate_MPDD(limit=1000, other_targets=other_targets)

**Example 2 — require mixed dimensionality across length scales.** Constrain the Larsen-style dimensionality descriptors at low, medium, and high cutoffs to all equal `[2, 0]`. This selects materials that are unambiguously two-dimensional with intercalated ions/molecules regardless of bond-cutoff choice (at least one 2D component and at least one 0D component).

In [ ]:
other_targets = {
    "material.dimensionalityRaw.larsenLow": [2, 0],
    "material.dimensionalityRaw.larsenMedium": [2, 0],
    "material.dimensionalityRaw.larsenHigh": [2, 0],
}
crystallSearch.generate_MPDD(limit=1000, other_targets=other_targets)

**Example 3 — restrict to a specific crystal system.** Pull only monoclinic prototypes from MPDD. Useful if you are only interested in systems of a particular kind.

In [ ]:
other_targets = {"material.crystalSystem": "monoclinic"}
crystallSearch.generate_MPDD(limit=1000, other_targets=other_targets)

Running the examples above will create new prototype folders in your working directory. So will see directories of the following form.

- `Ba1S1O4-MPDD(curated)-q<hash>-prototypes/` — three of these, one for each of Examples 1–3. The 6-character `q<hash>` is a deterministic short hash of the `other_targets` dict, which is how reruns with different filters avoid overwriting each other.
- `Ba1S1O4-MPDD(curated)-prototypes/` — If you ran an unfiltered `generate_MPDD(limit=1000)` call.

A couple things worth doing:

- **Identify which folder is which.** Next to each directory you will find a sibling `_query.json` file (e.g., `Ba1S1O4-MPDD(curated)-q3a91fc-prototypes_query.json`) recording the exact inputs that produced it — `target_formula`, `target_prototypes`, `filter_prototypes`, `collections`, `limit`, and `other_targets`. Open one to map a hash back to a human-readable filter without guesswork.
- **Spot-check structures.** Open a few `.POSCAR` files (e.g., in VESTA or ASE) to confirm the filter held — Example 1's outputs should look layered, Example 3's should be monoclinic, and so on.